# Network Classification and Link Prediction Analysis

This notebook reproduces **Figure 4**, **Figure 6**, **Table 2**, and **Table 3** from Sections 4.3, 4.4, and 4.5, which validate the relationship between multiscale entropy and network predictability.

## Overview

We perform two complementary analyses on the ICON dataset:

1. **Clustering Analysis (Figure 4, Table 2)**: K-means clustering (k=3) on five-dimensional entropy vectors [100%, 80%, 60%, 40%, 20%] reveals three groups aligned with entropy behaviors: hybrid (social), increasing (economic/technological), and stable (transportation/informational).

2. **Link Prediction Analysis (Figure 6, Table 3)**: We compute link prediction entropy using Jaccard and Adamic-Adar indices across reduction levels, demonstrating that structural compression entropy strongly correlates with network predictability across domains.

Together, these analyses establish multiscale entropy as both a classification tool and a predictor of link prediction performance.

## Imports and Setup

In [ ]:
pip install pandas==1.5.3
pip install sklearn

^C
ERROR: Operation cancelled by user
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))

/tmp/ipykernel_1441/912229180.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [ ]:
import os
import sys

##get current file directory
current_dir = os.path.dirname(os.path.abspath('__file__'))
##get the parent directory
parent_dir = os.path.dirname(current_dir)
# Add the src directory to sys.path
src_dir = os.path.join(parent_dir, '../')
sys.path.append(src_dir)

from algorithm.calculo_entropia import *
import algorithm.calculo_entropia

from algorithm.coarsening_utils import *
import algorithm.graph_utils
import algorithm.coarsening_utils as cu
from algorithm.coarsening_utils import plot_coarsening_vertical

import numpy as np
import scipy as sp

import matplotlib
import matplotlib.pylab as plt
from mpl_toolkits.mplot3d import Axes3D

import networkx as nx
import pygsp as gsp
from pygsp import graphs
gsp.plotting.BACKEND = 'matplotlib'

import pickle

def save_graphs(graph_dict, filename):
    """
    Save the dictionary of graphs to a file.
    """
    # Convert PyGSP graphs to NetworkX graphs for easier serialization
    nx_graph_dict = {
        size: [nx.from_scipy_sparse_array(g.W) for g in graphs]
        for size, graphs in graph_dict.items()
    }
    
    with open(filename, 'wb') as f:
        pickle.dump(nx_graph_dict, f)
    print(f"Graphs saved to {filename}")

def load_graphs(filename):
    """
    Load the dictionary of graphs from a file.
    """
    with open(filename, 'rb') as f:
        nx_graph_dict = pickle.load(f)
    print(f"Graphs loaded from {filename}")
    return nx_graph_dict

In [ ]:
import pandas as pd
print(pd.__version__)

1.5.3


In [ ]:
import pickle  
# load the data 
infile = open('./CommunityFitNet_updated.pickle','rb')  
df = pickle.load(infile) 

In [ ]:
# read edge lists for all networks
df_edgelists = df['edges_id'] # column 'edges_id' in dataframe df includes the edge list 
                              # for each network 
 
# extract the edge list for the first network 
edges_orig = df_edgelists.iloc[0] # a numpy array of edge list for original graph 


## Network Classification via Multiscale Entropy Clustering

To quantitatively validate the distinct entropy trajectory patterns observed across network domains, we perform unsupervised clustering using K-means (k=3) on the five-dimensional entropy vectors derived from each network's multiscale reduction sequence.

### Methodology

Each network is represented by a feature vector containing normalized compression entropy values at five reduction levels: [100%, 80%, 60%, 40%, 20%]. These vectors are standardized using z-score normalization before applying K-means clustering with k=3, chosen to align with the three dominant behavioral patterns identified earlier: **stable**, **increasing**, and **hybrid** entropy trajectories.

For visualization, we project the high-dimensional feature space onto its first two principal components using PCA. The clustering reveals clear spatial separation between groups, confirming that networks with similar multiscale entropy profiles naturally group together.

The results demonstrate a strong correspondence between network domain and cluster assignment. Social networks predominantly form one cluster (hybrid behavior), economic and technological networks another (increasing entropy), while transportation and informational networks group separately (stable entropy). Notably, biological networks are distributed across all clusters, reflecting the structural heterogeneity of biological systems.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import seaborn as sns
from collections import defaultdict
from sklearn.decomposition import PCA

# Cargar los datos
with open("graph_families_analysis.json", "r") as file:
    data = json.load(file)

# Extraer los valores de entropía normalizada de length compression
graphs = []  # Lista de nombres de grafos
features = []  # Lista para almacenar los vectores de 5 dimensiones
families = []

for family, graphs_data in data.items():
    for graph_name, graph_info in graphs_data.items():
        if "reductions" in graph_info:
            reductions = graph_info["reductions"]
            if all(str(p) in reductions for p in ["100", "80", "60", "40", "20"]):
                entropy_values = [
                    reductions[str(p)]["entropy_arithmetic"]["normalized"]
                    for p in [100, 80, 60, 40, 20]
                ]
                features.append(entropy_values)
                graphs.append(graph_name)
                families.append(family)

# Convertir a numpy array
X = np.array(features)

# Estandarizar los datos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Aplicar KMeans con 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X_scaled)
labels = kmeans.labels_
cluster_centers = kmeans.cluster_centers_

# Visualizar los clusters usando PCA para reducción a 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
centers_pca = pca.transform(cluster_centers)

plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=families, palette="tab10", s=100, alpha=0.8)
plt.scatter(centers_pca[:, 0], centers_pca[:, 1], c="red", marker="x", s=200, label="Cluster Center")

# Dibujar círculos alrededor de los clusters
for center in centers_pca:
    plt.gca().add_patch(plt.Circle(center, 0.5, color='gray', fill=False, linestyle='dashed'))

plt.title("Graph Clustering by Length Compression Normalized Entropy (KMeans, k=3)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend(title="Family Domain")
plt.show()

# Agrupar datos por cluster
cluster_composition = defaultdict(lambda: defaultdict(int))
for i, (family, graph_name) in enumerate(zip(families, graphs)):
    cluster_composition[labels[i]][family] += 1

# Mostrar la composición de los clusters
print("Composición de los clusters:\n")
for cluster, family_counts in sorted(cluster_composition.items()):
    print(f"Cluster {cluster}:")
    for family, count in family_counts.items():
        print(f"{family}: {count} grafos")
    print("")
